# Детектор автоматизированного трафика

В ноутбуке по шагам показаны проверка данных, построение признаков, временная валидация и финальное обучение.

Основные моменты:

1. события используются только внутри индивидуального окна `window_start_ts <= event_ts < window_end_ts`;
2. test расположен позже train, поэтому валидация делается по времени;
3. журнал событий агрегируется до уровня `cookie_id`;
4. качество оценивается по `Precision` при `Recall >= 70%`;
5. финальный score строится как смесь рангов обычного CatBoost и CatBoost с повышенным весом более свежих дней.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

# Ноутбук можно запускать как из корня задания, так и из папки solution/.
CWD = Path.cwd()
if (CWD / "data").exists():
    ROOT = CWD
    SOLUTION_DIR = CWD / "solution" if (CWD / "solution").exists() else CWD
elif (CWD.parent / "data").exists():
    ROOT = CWD.parent
    SOLUTION_DIR = CWD
else:
    raise FileNotFoundError("Не найдена папка data/. Запусти ноутбук из корня задания или из папки solution/.")

sys.path.insert(0, str(SOLUTION_DIR))

from feature_engineering import build_features, CATEGORICAL_FEATURES
from feature_order import FEATURE_ORDER
from metric import precision_at_recall
from train import (
    load_data,
    audit_data,
    run_validation,
    fit_and_predict,
    make_submission,
    make_base_model,
)

print("Корневая папка =", ROOT)
print("Папка решения =", SOLUTION_DIR)

## 1. Загрузка данных

Временные поля преобразуются в `datetime64` при чтении. Они нужны не только для признаков, но и для корректной фильтрации событий по окну наблюдения.


In [ ]:
train, test, events, sample_submission = load_data(ROOT)

print("Размер train:", train.shape)
print("Размер test:", test.shape)
print("Размер events:", events.shape)
print("Доля положительного класса:", train.target.mean())

display(train.head())
display(events.head())

## 2. Проверка данных и временных границ

Тестовые окна начинаются после обучающих, поэтому случайное разбиение здесь не очень хорошо повторяет реальный сценарий.

Отдельно проверяются события за пределами индивидуального окна cookie. Такие строки есть в исходном файле, но использовать их нельзя: к моменту формирования `score` они ещё неизвестны.


In [ ]:
audit_data(train, test, events, sample_submission)

In [ ]:
print("Распределение target по дням train:")
display(
    train.groupby(train.window_start_ts.dt.date).target
         .agg(["count", "sum", "mean"])
         .style.format({"mean": "{:.3f}"})
)

print("Типы событий:")
display(events.event_name.value_counts().to_frame("количество"))

## 3. Построение признаков

Функция `build_features` не обращается к `target`, поэтому один и тот же код используется для обучающей и тестовой выборок.

Основные группы признаков:

- количество и плотность событий;
- интервалы между соседними событиями;
- типы событий и переходы между ними;
- разнообразие объявлений, категорий, локаций и запросов;
- глубина просмотра поисковой выдачи;
- User-Agent, платформа и их сочетания;
- координаты указателя и параметры перемещения;
- возраст cookie и несколько календарных признаков;
- дубликаты и всплески событий в одну секунду.

Перед расчётом последовательностных признаков события обязательно сортируются по времени.


In [ ]:
X_train = build_features(events, train)
X_test = build_features(events, test)

assert X_train.columns.tolist() == X_test.columns.tolist()
assert len(FEATURE_ORDER) == X_train.shape[1] - 1

print("Размер матрицы признаков train:", X_train.shape)
print("Размер матрицы признаков test:", X_test.shape)
display(X_train.iloc[:5, :20])

## 4. Временная валидация

Одного последнего holdout оказалось недостаточно: основная метрика заметно меняется между соседними временными срезами. Поэтому в текущей версии используется rolling temporal validation на двух последних интервалах:

- `train < 2026-04-16`, validation — 16–18 апреля;
- `train < 2026-04-17`, validation — 17–19 апреля.

Для каждого фолда сравниваются обычный CatBoost, CatBoost с recency-весами и их ранговое смешивание.

In [ ]:
validation_report = run_validation(train, X_train)
display(validation_report.style.format({
    "P@R>=0.70": "{:.4f}",
    "PR-AUC": "{:.4f}",
    "ROC-AUC": "{:.4f}",
}))

### Основная метрика и PR-AUC

`PR-AUC` и `ROC-AUC` оставлены как диагностические показатели. Финальный выбор делается по `P@R>=0.70`, потому что именно эта метрика используется при проверке решения.


## 5. Важность признаков

Ниже CatBoost отдельно обучается на временной обучающей части, после чего выводятся 30 признаков с наибольшей важностью. Это диагностический шаг: важность дерева не следует трактовать как причинную зависимость.


In [ ]:
from catboost import CatBoostClassifier

is_valid = train.window_start_ts.ge("2026-04-17").to_numpy()
y = train.target.to_numpy()
cat_indices = [FEATURE_ORDER.index(c) for c in CATEGORICAL_FEATURES]

model = make_base_model()
model.fit(
    X_train.loc[~is_valid, FEATURE_ORDER],
    y[~is_valid],
    cat_features=cat_indices,
    verbose=False,
)

importance = (
    pd.DataFrame({
        "признак": FEATURE_ORDER,
        "важность": model.get_feature_importance(),
    })
    .sort_values("важность", ascending=False)
    .head(30)
)
display(importance)

## 6. Финальное обучение

После выбора схемы модель обучается на всей размеченной выборке. Порог в submission не записывается — нужен только непрерывный `score`.

Финальный вариант использует два CatBoost:

- основной — одинаковый вес всех train-объектов;
- recency-модель — более свежие дни имеют больший вес.

Итоговый score — смесь их нормированных рангов в пропорции 70/30.

In [ ]:
final_score = fit_and_predict(
    train=train,
    X_train=X_train,
    X_test=X_test,
    base_only=False,
)

submission = make_submission(
    test=test,
    sample=sample_submission,
    score=final_score,
    output=SOLUTION_DIR / "submission.csv",
)

display(submission.head())
print(submission.score.describe())

## 7. Краткое резюме

Основной риск в задаче — временная утечка: часть событий в исходном журнале находится после `window_end_ts`, поэтому фильтрация выполняется до любых агрегатов.

Второй важный момент — временной дрейф. Один последний holdout оказался нестабильным, поэтому для проверки используется несколько последовательных временных срезов. Финальная модель не смешивается с отдельным XGBoost: вместо этого используются две версии CatBoost, одна из которых сильнее учитывает свежие наблюдения.

`captcha_shown` в признаки не входит, потому что в предоставленных данных такие события встречаются за пределами разрешённого окна наблюдения.